# Invoice Categorization ML Project

This notebook performs step-by-step image categorization for invoices using machine learning.

## 1. Import Required Libraries

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## 2. Load and Preprocess Dataset

In [ ]:
# Define paths
dataset_path = 'dataset'
processed_path = 'processed_data'
os.makedirs(processed_path, exist_ok=True)

# Function to preprocess image
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return None
    # Resize to 224x224
    image = cv2.resize(image, (224, 224))
    return image

# Collect all image paths and labels
image_paths = []
labels = []
categories = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(('.jpg', '.png', '.jpeg')):
            image_path = os.path.join(root, file)
            category = os.path.basename(root)
            if category not in categories:
                categories.append(category)
            label = categories.index(category)
            image_paths.append(image_path)
            labels.append(label)

print(f"Found {len(image_paths)} images in {len(categories)} categories: {categories}")

# Split into train/val/test
X_train, X_temp, y_train, y_temp = train_test_split(image_paths, labels, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Data generators
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(processed_path, 'train'),
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    classes=categories
) if os.path.exists(os.path.join(processed_path, 'train')) else None

# If processed data exists, use it; else, process on the fly (for demo, assume processed)
# For simplicity, assume data is processed, or run prepare_data.py first

## 3. Build the Neural Network Model

In [ ]:
num_classes = len(categories)

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.summary()

## 4. Compile the Model

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

## 5. Train the Model

In [ ]:
# Note: Run prepare_data.py first to generate processed_data/train, val, test folders
if train_generator:
    history = model.fit(
        train_generator,
        epochs=10,
        validation_data=val_generator
    )
else:
    print("Processed data not found. Run prepare_data.py first.")

## 6. Evaluate Model Performance

In [ ]:
if 'history' in locals():
    test_loss, test_acc = model.evaluate(test_generator)
    print(f'Test accuracy: {test_acc}')
    
    # Plot training history
    plt.plot(history.history['accuracy'], label='accuracy')
    plt.plot(history.history['val_accuracy'], label='val_accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.show()
else:
    print("Model not trained yet.")

## 7. Make Predictions on New Images

In [ ]:
def predict_image(image_path):
    image = preprocess_image(image_path)
    if image is not None:
        image = np.expand_dims(image, axis=0) / 255.0
        prediction = model.predict(image)
        predicted_class = np.argmax(prediction)
        return categories[predicted_class]
    return None

# Example prediction
# predicted_category = predict_image('path/to/new/invoice.jpg')
# print(f'Predicted category: {predicted_category}')